In [5]:
!pip install flash-linear-attention

In [6]:
import torch
from fla.layers.simple_gla import SimpleGatedLinearAttention as GLA

In [16]:
import torch
import time


BATCH_SIZE = 2
SEQ_LEN = 4096
HIDDEN_SIZE = 128
NUM_HEADS = 4
DTYPE = torch.float16
DEVICE = 'cuda'

model = GLA(
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    mode='chunk', 
    use_short_conv=True
)

model = model.to(device=DEVICE, dtype=DTYPE)
model.eval()

x = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE)

print(f"Benchmarking SimpleGatedLinearAttention ({model.mode} mode)...")
print(f"Input Shape: {x.shape}")

print("Warming up GPU...")
for _ in range(10):
    with torch.no_grad():
        _ = model(x)
torch.cuda.synchronize()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

start_event.record()

n_loops = 100
for _ in range(n_loops):
    with torch.no_grad():
        y, _, _ = model(x)

end_event.record()
torch.cuda.synchronize()

elapsed_time_ms = start_event.elapsed_time(end_event)
avg_time = elapsed_time_ms / n_loops

print(f"Output shape: {y.shape}")
print(f"Total time for {n_loops} runs: {elapsed_time_ms:.2f} ms")
print(f"Average time per forward pass: {avg_time:.4f} ms")

Benchmarking SimpleGatedLinearAttention (chunk mode)...
Input Shape: torch.Size([2, 4096, 128])
Warming up GPU...


Output shape: torch.Size([2, 4096, 128])
Total time for 100 runs: 176.97 ms
Average time per forward pass: 1.7697 ms
